**Question:** Does adding ELO at the GRU head improve on the ELO-free GRU?

**Assumptions:** Architecture, split and hyperparameters identical to Exp C; only ELO scalars are added.

**Method:** Train Exp C2 (GRU + ELO) and compare it with C, B' and ELO.

**Result:** C2 reaches Brier 0.1609 (vs C 0.1628), a small gain over C that is significant against ELO.

# 08 — Experiment C2: GRU + ELO ("Neural ELO, ELO-Included")

Follow-up to
`05_gru_neural_elo.ipynb` and `06_confidence_intervals_cv.ipynb`. Exp C (the ELO-free GRU,
calibrated test Brier 0.16278) was deliberately built without ever seeing an `EloBefore` number.
**This notebook adds point-in-time ELO to the GRU** — the direct parallel of Exp B (ELO added to
XGBoost) — to answer: does giving the GRU the ELO number on top of its own learned sequence
representation make it the outright best model, or is ELO redundant once the GRU has learned its
own skill representation from the raw win/loss + razina-trajectory sequence?

**How ELO is injected (Step 8 below):** the exact same architecture, data pipeline, `MAX_LEN=24`
sequences, 7 per-step features, duplicate-and-flip symmetric expansion, 80/20 time split, and
train-carved calib/early-stop slices as Exp C — **unchanged**. The only addition: each player's
own point-in-time `EloBefore` (`player_elo`/`opponent_elo` from `features.parquet` — the SAME
values Exp 0/B use, reused directly rather than recomputed) is **concatenated at the head**, so
the MLP sees `[h_player, h_opponent, h_player − h_opponent, elo_player_scaled, elo_opponent_scaled]`
instead of Exp C's 3-way concat alone. This is head-level ELO injection (as opposed to adding
ELO as a per-step sequence feature, which Exp C3 does later) — the direct parallel to how Exp B adds `elo_diff` as one extra XGBoost feature,
except here both the player's and the opponent's own scaled ELO are kept as two separate scalars
(not collapsed to a single diff) so the head can use both relative AND absolute skill level.
By design, only ONE variant is trained (head-level; no separate per-step-feature variant) —
this pipeline was validated in a standalone script first (see inline notes) before being assembled
here.

**Train/test hygiene:** the ELO scaling constants (`ELO_MEAN`, `ELO_STD`) are computed from
TRAIN ONLY (`Split == 'train'` rows of `features.parquet`), never from TEST — same discipline as
`05`'s `LIGAKOEF_SCALE` fix.

**Reused, unchanged from Exp C / Exp A/B:**
- `matches_export.csv` + `features.parquet`'s `Split` column (train 2017-05-08..2025-01-18 / test
  2025-01-19..2026-07-08, 3,426 test matches / 6,852 duplicate-and-flip rows).
- Duplicate-and-flip symmetric expansion (symmetry guaranteed by construction, not an
  antisymmetric head).
- `calib_frac=0.15`, `early_stop_frac=0.10` chronological TRAIN-only carve.
- `TwoTowerGRU` architecture (shared-weight GRU, `hidden_size=24`, `dropout=0.3`), AdamW
  (`lr=2e-3`, `weight_decay=1e-4`), `BATCH_SIZE=512`, `MAX_EPOCHS=60`, `PATIENCE=8`, seed=42.
- Calibration: temperature scaling vs isotonic regression on the calib slice, whichever wins on
  the calib slice itself is applied to TEST (same as Exp C).
- Evaluation: accuracy / Brier (primary) / log-loss / AUC on the identical 6,852-row TEST set,
  PLUS match-level bootstrap CIs (2,000 resamples, seed 42) and paired-bootstrap Brier
  differences — same methodology as `06_confidence_intervals_cv.ipynb` Part 1/2.
- **No walk-forward CV in this notebook** (single split + bootstrap only, by design).


In [ ]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(False)  # GRU has no deterministic CPU kernel path for this op set

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cpu')
print(f'PyTorch {torch.__version__}, device={device}, threads={torch.get_num_threads()}')


## Step 1 — Load raw export + the P2 engineered frame (identical to Exp C)

`features.parquet` supplies `PlayerId`/`OpponentId`/`Label`/`Split`/`Datum`/`SusretId` (the
duplicate-and-flip row set and time split) — **and now also `player_elo`/`opponent_elo`**, the
point-in-time `EloBefore` values this notebook injects at the GRU head.

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)

sym = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)

print(f'raw matches: {raw.shape}')
print(f'sym (duplicate-and-flip) frame: {sym.shape}')
print(f"Split date (from P2): {TAGS['split_date']}, train matches: {TAGS['n_matches_train']:,}, "
      f"test matches: {TAGS['n_matches_test']:,}")
sym[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective', 'Split', 'player_elo', 'opponent_elo']].head()


## Step 2 — Long-format player-match log (identical to Exp C)

One row per (player, match), same construction as `05_gru_neural_elo.ipynb` Step 2.

In [ ]:
common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']

winner_rows = raw[common_cols + ['WinnerId', 'LoserId', 'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
             'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
             'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3'}
).assign(Won=1)

loser_rows = raw[common_cols + ['LoserId', 'WinnerId', 'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
             'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
             'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3'}
).assign(Won=0)

player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
# CRITICAL: strict ascending (IgracId, Datum, SusretId) order -- every point-in-time feature
# below only ever looks at rows ABOVE the current one within a player's own group, i.e.
# strictly BEFORE this match. Same discipline as notebook 02/05's SQL ROW_NUMBER pattern.
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
print(f'player_log: {player_log.shape}')
player_log.head(3)


## Step 3 — Per-step point-in-time features (identical to Exp C — still ELO-free per step)

Note: the per-step sequence features fed to the GRU stay exactly as ELO-free as Exp C — `Won`,
`SetMargin`, `DaysSinceLastMatch`, `LigaKoeficjent` (razina), `HomeShareBefore`,
`OpponentCareerWinRateBefore`. ELO is injected ONLY once, at the head, as the two new scalars
built in Step 6 below — the sequence itself is untouched.

In [ ]:
g = player_log.groupby('IgracId', sort=False)
player_log['MatchCountBefore'] = g.cumcount()
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore'],
    np.nan,
)
player_log['DaysSinceLastMatch'] = player_log.groupby('IgracId')['Datum'].diff().dt.days
# ELO-free: built exclusively from the win/loss graph, never from EloBefore/EloAfter.
player_log['CareerWinRateBefore'] = g['Won'].transform(lambda s: s.shift().expanding().mean())

for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']:
    player_log[c + '_f'] = player_log[c].fillna(0)
player_log['SetMargin'] = (
    (player_log['MySet1_f'] + player_log['MySet2_f'] + player_log['MySet3_f'])
    - (player_log['OppSet1_f'] + player_log['OppSet2_f'] + player_log['OppSet3_f'])
)
player_log = player_log.drop(columns=[c + '_f' for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']])

check_w = raw[['SusretId', 'WinnerId', 'WinnerMatchCountBefore']].rename(columns={'WinnerId': 'IgracId', 'WinnerMatchCountBefore': 'Exp'})
check_l = raw[['SusretId', 'LoserId', 'LoserMatchCountBefore']].rename(columns={'LoserId': 'IgracId', 'LoserMatchCountBefore': 'Exp'})
chk = pd.concat([check_w, check_l])
mm = player_log.merge(chk, on=['SusretId', 'IgracId'], how='left')
mismatches = (mm['MatchCountBefore'] != mm['Exp']).sum()
assert mismatches == 0, 'pandas point-in-time ordering does not match the SQL export!'
print(f'MatchCountBefore cross-check vs SQL export: {mismatches} mismatches -- OK.')
player_log[['Won', 'SetMargin', 'DaysSinceLastMatch', 'LigaKoeficjent', 'HomeShareBefore', 'CareerWinRateBefore']].describe()


## Step 4 — Opponent-quality-at-that-time proxy (identical to Exp C)

Same self-merge on `(SusretId, OpponentId)` as `05` — ELO-free by construction.

In [ ]:
opp_cwr = player_log[['SusretId', 'IgracId', 'CareerWinRateBefore']].rename(
    columns={'IgracId': 'OpponentId', 'CareerWinRateBefore': 'OpponentCareerWinRateBefore'})
player_log = player_log.merge(opp_cwr, on=['SusretId', 'OpponentId'], how='left', validate='many_to_one')
print(f"OpponentCareerWinRateBefore NaN share (opponent's own debut match): "
      f"{player_log['OpponentCareerWinRateBefore'].isna().mean():.2%}")
player_log[['OpponentCareerWinRateBefore']].describe()


## Step 5 — Build the per-step feature array + history precomputation (identical to Exp C)

`MAX_LEN=24`, 7-dim per-step vector, debut placeholder step — same as `05`. `LIGAKOEF_SCALE` is
computed TRAIN-only (the hygiene fix already established and verified inert in `05` Step 15).

In [ ]:
MAX_LEN = 24
N_FEATS = 7  # Won, SetMarginScaled, DaysRestScaled, LigaKoefScaled, HomeShareBefore, OppCareerWinRateBefore, IsPlaceholder

SETMARGIN_SCALE = 12.0
DAYSREST_CLIP = 90.0

_split_date = pd.Timestamp(TAGS['split_date'])
_train_mask_pl = player_log['Datum'] < _split_date
LIGAKOEF_SCALE = float(player_log.loc[_train_mask_pl, 'LigaKoeficjent'].max())  # train-only max
print(f"LIGAKOEF_SCALE (train-only) = {LIGAKOEF_SCALE}")

step_feats = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
step_feats[:, 0] = player_log['Won'].values
step_feats[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
step_feats[:, 2] = days_rest / DAYSREST_CLIP
step_feats[:, 3] = player_log['LigaKoeficjent'].values / LIGAKOEF_SCALE
step_feats[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
step_feats[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
step_feats[:, 6] = 0.0  # real step

PLACEHOLDER_VEC = np.zeros(N_FEATS, dtype=np.float32)
PLACEHOLDER_VEC[5] = 0.5
PLACEHOLDER_VEC[6] = 1.0

player_log['_gpos'] = np.arange(len(player_log))
first_idx = player_log.groupby('IgracId')['_gpos'].min().to_dict()
idx_lookup = dict(zip(zip(player_log['IgracId'], player_log['SusretId']), player_log['_gpos']))

def build_history(igrac_id, susret_id):
    gpos = idx_lookup[(igrac_id, susret_id)]
    start = first_idx[igrac_id]
    p = gpos - start
    seq = np.zeros((MAX_LEN, N_FEATS), dtype=np.float32)
    if p == 0:
        seq[0] = PLACEHOLDER_VEC
        length = 1
    else:
        lo = max(start, gpos - MAX_LEN)
        chunk = step_feats[lo:gpos]
        length = chunk.shape[0]
        seq[:length] = chunk
    return seq, length

n_rows = len(player_log)
HIST_ARRAY = np.zeros((n_rows, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
for i in range(n_rows):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])

print(f'Precomputed histories for {n_rows:,} player-match rows.')
print(f'Sequence length -- min={LEN_ARRAY.min()}, median={np.median(LEN_ARRAY):.0f}, max={LEN_ARRAY.max()}')


### Leakage spot-check (mandatory — reapplied here)

Same discipline as `05` Step 12: `p` (position from the global sort) must exactly equal
`MatchCountBefore`, and a concrete TEST-period player's history window must be strictly before
their own predicted match date.

In [ ]:
p_check = (player_log['_gpos'] - player_log['IgracId'].map(first_idx))
assert (p_check == player_log['MatchCountBefore']).all(), 'group contiguity assumption violated!'
print('OK -- p (derived from global sort) == MatchCountBefore for all rows (group contiguity holds).')

test_period_start = pd.Timestamp(TAGS['split_date'])
sample = player_log.loc[player_log['Datum'] >= test_period_start].iloc[100]
gpos, start = sample['_gpos'], first_idx[sample['IgracId']]
hist_rows = player_log.iloc[start:gpos]
assert (hist_rows['Datum'] < sample['Datum']).all(), 'LEAKAGE: a history row is not strictly prior!'
print(f"OK -- spot-checked IgracId={sample['IgracId']}, SusretId={sample['SusretId']}: "
      f"every history row is strictly before the predicted match date. No leakage.")


## Step 6 — Resolve `features.parquet` rows onto history positions + chronological carve

Identical to Exp C Steps 6-7: `core` (fit) / `eval` (early-stopping) / `calib` (post-hoc
calibration) / `test`, same `calib_frac=0.15`, `early_stop_frac=0.10` carve at MATCH level.

In [ ]:
sym = sym.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym['PlayerId'], sym['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym['OpponentId'], sym['SusretId'])])
print(f'Resolved player/opponent history positions for {len(sym):,} rows.')

train_matches = (
    sym.loc[sym['Split'] == 'train', ['SusretId', 'Datum']]
    .drop_duplicates()
    .sort_values(['Datum', 'SusretId'], kind='mergesort')
    .reset_index(drop=True)
)
n_train_matches = len(train_matches)
CALIB_FRAC, EARLY_STOP_FRAC = 0.15, 0.10

n_calib = int(round(n_train_matches * CALIB_FRAC))
fit_order = train_matches.iloc[: n_train_matches - n_calib].reset_index(drop=True)
calib_ids = set(train_matches.iloc[n_train_matches - n_calib:]['SusretId'])

n_fit = len(fit_order)
n_eval = int(round(n_fit * EARLY_STOP_FRAC))
core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])

sym['Bucket'] = np.where(sym['Split'] == 'test', 'test',
                  np.where(sym['SusretId'].isin(calib_ids), 'calib',
                  np.where(sym['SusretId'].isin(eval_ids), 'eval', 'core')))
print(sym['Bucket'].value_counts())
assert sym.loc[sym['Bucket'] == 'test'].shape[0] == TAGS['n_matches_test'] * 2


## Step 7 — NEW: point-in-time ELO scalars for the head

`player_elo` / `opponent_elo` are already present in `sym` (== `features.parquet`) — these are
the exact same point-in-time `EloBefore` values Exp 0/B use (`elo_diff = player_elo −
opponent_elo`). Scaled here via **TRAIN-only** mean/std (pooled over both `player_elo` and
`opponent_elo` train rows, since the two columns share an identical marginal distribution by the
duplicate-and-flip construction) — the same train/test hygiene discipline as `05`'s
`LIGAKOEF_SCALE` fix. A GRU's head-layer weights can absorb a fixed affine rescale, so this
particular choice of center/scale is not expected to matter much, but computing it from TRAIN
only (never TEST) is the correct discipline regardless.

In [ ]:
train_elo_mask = sym['Split'] == 'train'
ELO_MEAN = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).mean())
ELO_STD = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).std())
print(f'ELO_MEAN (train-only) = {ELO_MEAN:.2f}')
print(f'ELO_STD (train-only)  = {ELO_STD:.2f}')

sym['elo_p_scaled'] = (sym['player_elo'] - ELO_MEAN) / ELO_STD
sym['elo_o_scaled'] = (sym['opponent_elo'] - ELO_MEAN) / ELO_STD
sym[['player_elo', 'opponent_elo', 'elo_p_scaled', 'elo_o_scaled']].describe()


## Step 8 — Two-tower shared-weight GRU model + ELO-at-head (Exp C2)

Identical `TwoTowerGRU` from Exp C (`hidden_size=24`, dropout 0.3 on GRU output + inside MLP
head), **with 2 extra scalar inputs (`elo_p_scaled`, `elo_o_scaled`) concatenated into the MLP
head** alongside `[h_p, h_o, h_p − h_o]` — so the head's input width grows from `3×24=72` to
`74`. Everything else (shared GRU weights across both towers, symmetry-by-duplicate-and-flip,
`BCEWithLogitsLoss`, AdamW weight decay `1e-4`, dropout `0.3`, `MAX_LEN=24`, early stopping on
chronological `eval`-bucket Brier, seed 42) is unchanged from Exp C.

In [ ]:
HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)


class TwoTowerGRUElo(nn.Module):
    """Identical TwoTowerGRU architecture from Exp C, with 2 extra scalar inputs
    (elo_p_scaled, elo_o_scaled) concatenated into the MLP head -- the direct parallel of
    Exp B's elo_diff feature added to XGBoost."""

    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + 2, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, elo_p, elo_o):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, elo_p.unsqueeze(1), elo_o.unsqueeze(1)], dim=1)
        return self.head(combined).squeeze(-1)


def bucket_tensors(bucket_name):
    mask = (sym['Bucket'] == bucket_name).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    ep = torch.as_tensor(sym.loc[mask, 'elo_p_scaled'].values, dtype=torch.float32)
    eo = torch.as_tensor(sym.loc[mask, 'elo_o_scaled'].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y, ep, eo

core_pseq, core_plen, core_oseq, core_olen, core_y, core_ep, core_eo = bucket_tensors('core')
eval_pseq, eval_plen, eval_oseq, eval_olen, eval_y, eval_ep, eval_eo = bucket_tensors('eval')
calib_pseq, calib_plen, calib_oseq, calib_olen, calib_y, calib_ep, calib_eo = bucket_tensors('calib')
test_pseq, test_plen, test_oseq, test_olen, test_y, test_ep, test_eo = bucket_tensors('test')
print(f'core={core_y.shape[0]:,}  eval={eval_y.shape[0]:,}  calib={calib_y.shape[0]:,}  test={test_y.shape[0]:,}')


## Step 9 — Training loop (identical hyperparameters + early stopping to Exp C)

Same manual full-tensor mini-batches, same `BATCH_SIZE=512`, `MAX_EPOCHS=60`, `PATIENCE=8`, early
stopping on the chronological `eval` bucket's Brier score (never TEST, never `calib`).

In [ ]:
HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8

torch.manual_seed(RANDOM_STATE)
model = TwoTowerGRUElo(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
n_core = core_y.shape[0]

def eval_bucket(p_seq, p_len, o_seq, o_len, ep, eo, y):
    model.eval()
    with torch.no_grad():
        logit = model(p_seq, p_len, o_seq, o_len, ep, eo)
    return brier_score_loss(y.numpy(), torch.sigmoid(logit).numpy()), logit

best_brier, best_state, epochs_no_improve = np.inf, None, 0
history, rng = [], np.random.RandomState(RANDOM_STATE)

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    perm = rng.permutation(n_core)
    total_loss, n_batches = 0.0, 0
    for start in range(0, n_core, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        logit = model(core_pseq[idx], core_plen[idx], core_oseq[idx], core_olen[idx],
                      core_ep[idx], core_eo[idx])
        loss = criterion(logit, core_y[idx])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    val_brier, _ = eval_bucket(eval_pseq, eval_plen, eval_oseq, eval_olen, eval_ep, eval_eo, eval_y)
    history.append({'epoch': epoch, 'train_loss': total_loss / n_batches, 'val_brier': val_brier})

    improved = val_brier < best_brier - 1e-5
    if improved:
        best_brier, best_state, epochs_no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or improved or epoch == 1:
        print(f'epoch {epoch:3d}  train_loss={total_loss/n_batches:.4f}  val_brier={val_brier:.5f}'
              f"{'  <- best' if improved else ''}")
    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no improvement in {PATIENCE} epochs).')
        break

model.load_state_dict(best_state)
print(f'\nTraining done in {time.time()-t0:.1f}s. Best eval-bucket Brier = {best_brier:.5f}')
history_df = pd.DataFrame(history)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df['epoch'], history_df['train_loss'], label='train loss (BCE, core)', color='#2563eb')
ax2 = ax.twinx()
ax2.plot(history_df['epoch'], history_df['val_brier'], label='val Brier (eval bucket)', color='#dc2626')
ax.set_xlabel('epoch'); ax.set_ylabel('train BCE loss', color='#2563eb'); ax2.set_ylabel('val Brier', color='#dc2626')
ax.set_title('Exp C2 training curve (GRU + ELO-at-head) -- early stopping on chronological eval-bucket Brier')
fig.tight_layout()
plt.savefig('../models/exp_c2_training_curve.png', dpi=110)
plt.show()


## Step 10 — Calibration on the `calib` bucket (identical methodology to Exp C)

Temperature scaling vs isotonic regression, both fit on the SAME calib slice; whichever wins on
the calib slice itself is applied to TEST.

In [ ]:
model.eval()
with torch.no_grad():
    calib_logit = model(calib_pseq, calib_plen, calib_oseq, calib_olen, calib_ep, calib_eo)
    test_logit = model(test_pseq, test_plen, test_oseq, test_olen, test_ep, test_eo)

calib_logit_np, calib_y_np = calib_logit.numpy(), calib_y.numpy()
test_logit_np, test_y_np = test_logit.numpy(), test_y.numpy()

T_param = torch.nn.Parameter(torch.ones(1))
calib_logit_t, calib_y_t = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

def _closure():
    opt_T.zero_grad()
    loss = nn.functional.binary_cross_entropy_with_logits(calib_logit_t / T_param, calib_y_t)
    loss.backward()
    return loss

opt_T.step(_closure)
T_value = float(T_param.detach().clamp(min=1e-2))

p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
p_calib_temp = torch.sigmoid(calib_logit_t / T_value).numpy()

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib_raw, calib_y_np)
p_calib_iso = iso.predict(p_calib_raw)

brier_raw = brier_score_loss(calib_y_np, p_calib_raw)
brier_temp = brier_score_loss(calib_y_np, p_calib_temp)
brier_iso = brier_score_loss(calib_y_np, p_calib_iso)
print(f'Fitted temperature T = {T_value:.4f}')
print(f'Calib-slice Brier -- raw: {brier_raw:.5f}, temperature: {brier_temp:.5f}, isotonic: {brier_iso:.5f}')

p_test_raw = 1 / (1 + np.exp(-test_logit_np))
if brier_iso < brier_temp:
    calib_method = 'isotonic'
    p_test_calibrated = iso.predict(p_test_raw)
else:
    calib_method = 'temperature'
    p_test_calibrated = torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()
print(f'Chosen calibration method for TEST: {calib_method}')


## Step 11 — Score TEST (identical 6,852-row set) — full comparison: Exp 0/A/A'/B/B'/C/C2

Exp 0/A/B/C predictions come from `test_predictions_4way.parquet` (`03`/`05`); Exp A'/B' from
`test_predictions_trajectory.parquet` (`07`) — all row-set-verified to be the exact same TEST
rows before comparing.

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

preds4 = pd.read_parquet(DATA_DIR / 'test_predictions_4way.parquet')
preds_traj = pd.read_parquet(DATA_DIR / 'test_predictions_trajectory.parquet')

test_bucket_keys = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId', 'Label']]
merge_check = test_bucket_keys.merge(preds4[['SusretId', 'PlayerId', 'OpponentId', 'Label']],
                                      on=['SusretId', 'PlayerId', 'OpponentId'], suffixes=('_c2', '_exp'))
assert len(merge_check) == len(test_bucket_keys) == len(preds4), 'Row-set mismatch vs Exp 0/A/B/C test set!'
assert (merge_check['Label_c2'] == merge_check['Label_exp']).all(), 'Label mismatch vs Exp 0/A/B/C test set!'
print(f'OK -- Exp C2 test bucket is the exact same {len(preds4):,}-row set as Exp 0/A/B/C, labels match.')

elo_brier = brier_score_loss(preds4['Label'], preds4['p_exp0_elo'])
comparison = pd.DataFrame([
    compute_metrics(preds4['Label'].values, preds4['p_exp0_elo'].values, 'Exp 0 - ELO baseline'),
    compute_metrics(preds4['Label'].values, preds4['p_expA_calibrated'].values, "Exp A - ELO-free XGBoost"),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expAprime_calibrated'].values, "Exp A' - ELO-free XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expB_calibrated'].values, "Exp B - ELO-included XGBoost"),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expBprime_calibrated'].values, "Exp B' - ELO-included XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expC_calibrated'].values, 'Exp C - ELO-free GRU'),
    compute_metrics(test_y_np, p_test_calibrated, 'Exp C2 - GRU + ELO (calibrated)'),
]).set_index('experiment')
comparison['brier_delta_vs_elo_pct'] = (comparison['brier'] - elo_brier) / elo_brier * 100
comparison_display = comparison.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
comparison_display


In [ ]:
print('Headline point-estimate verdict:')
for exp in comparison.index:
    d = comparison.loc[exp, 'brier_delta_vs_elo_pct']
    print(f"  {exp}: Brier {d:+.2f}% vs ELO baseline ({'WORSE' if d > 0 else 'better'} -- higher Brier = worse).")

# Raw (uncalibrated) vs calibrated GRU+ELO -- same calibration-effect sanity check as Exp C
raw_metrics = compute_metrics(test_y_np, p_test_raw, 'Exp C2 - raw (uncalibrated)')
cal_metrics = compute_metrics(test_y_np, p_test_calibrated, 'Exp C2 - calibrated')
pd.DataFrame([raw_metrics, cal_metrics]).set_index('experiment').round(5)


## Step 12 — Match-level bootstrap CIs + paired-bootstrap Brier differences

Identical methodology to `06_confidence_intervals_cv.ipynb` Part 1/2 and `07`'s Step 7/8: 2,000
resamples, seed 42, both perspective rows of a match always travel together (drawing match IDs
with replacement, not individual rows). No walk-forward CV here (single split + bootstrap only,
by design) — see `06`'s written summary for the walk-forward caveat on how C-vs-B
rankings can be fold-dependent; that caveat should be read alongside this notebook's C2-vs-B(')
comparisons below.

In [ ]:
preds_all = preds4.merge(preds_traj[['SusretId', 'PlayerId', 'OpponentId', 'p_expAprime_calibrated', 'p_expBprime_calibrated']],
                          on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
exp_c2_preds = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId']].copy()
exp_c2_preds['p_expC2_calibrated'] = p_test_calibrated
merged = preds_all.merge(exp_c2_preds, on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
assert len(merged) == len(preds4) == len(preds_traj), 'row-set mismatch across prediction files!'
print(f'Merged all 7 experiments onto {len(merged):,} identical test rows.')

EXPERIMENTS = {
    'Exp 0 - ELO baseline':                   'p_exp0_elo',
    'Exp A - ELO-free XGB':                   'p_expA_calibrated',
    "Exp A' - ELO-free XGB + trajectory":     'p_expAprime_calibrated',
    'Exp B - ELO-included XGB':               'p_expB_calibrated',
    "Exp B' - ELO-included XGB + trajectory": 'p_expBprime_calibrated',
    'Exp C - ELO-free GRU':                   'p_expC_calibrated',
    'Exp C2 - GRU + ELO':                     'p_expC2_calibrated',
}
y_all = merged['Label'].values
p_all = {name: merged[col].values for name, col in EXPERIMENTS.items()}

def m_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
           'brier': brier_score_loss(y_true, p), 'log_loss': log_loss(y_true, p_c)}
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out

point_estimates = {name: m_dict(y_all, p_all[name]) for name in EXPERIMENTS}

groups = merged.groupby('SusretId').indices
match_ids = np.array(list(groups.keys()))
lens = np.array([len(groups[m]) for m in match_ids])
assert (lens == 2).all(), 'expected exactly 2 rows (both perspectives) per match'
row_idx_2d = np.array([groups[m] for m in match_ids], dtype=np.int64)
n_matches = len(match_ids)
print(f'TEST set: {len(merged):,} rows, {n_matches:,} unique matches.')


In [ ]:
N_BOOT = 2000
rng = np.random.RandomState(RANDOM_STATE)

boot_metrics = {name: {'accuracy': [], 'brier': [], 'log_loss': [], 'auc': []} for name in EXPERIMENTS}
DIFF_KEYS = {
    'C2_minus_C': ('Exp C2 - GRU + ELO', 'Exp C - ELO-free GRU'),
    'C2_minus_B': ('Exp C2 - GRU + ELO', 'Exp B - ELO-included XGB'),
    "C2_minus_Bprime": ('Exp C2 - GRU + ELO', "Exp B' - ELO-included XGB + trajectory"),
    'C2_minus_0': ('Exp C2 - GRU + ELO', 'Exp 0 - ELO baseline'),
}
boot_diff = {k: [] for k in DIFF_KEYS}

t0 = time.time()
for b in range(N_BOOT):
    sampled_match_pos = rng.randint(0, n_matches, size=n_matches)
    idx = row_idx_2d[sampled_match_pos].reshape(-1)
    y_b = y_all[idx]

    briers = {}
    for name in EXPERIMENTS:
        p_b = p_all[name][idx]
        m = m_dict(y_b, p_b)
        for k in boot_metrics[name]:
            boot_metrics[name][k].append(m[k])
        briers[name] = m['brier']

    for dk, (a, b_) in DIFF_KEYS.items():
        boot_diff[dk].append(briers[a] - briers[b_])

print(f'Match-level bootstrap done: {N_BOOT:,} resamples in {time.time()-t0:.1f}s (seed={RANDOM_STATE})')


### 95% Bootstrap CI table (all 7 experiments)

In [ ]:
rows = []
for name in EXPERIMENTS:
    row = {'experiment': name}
    for metric in ['accuracy', 'brier', 'log_loss', 'auc']:
        arr = np.array(boot_metrics[name][metric])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        row[f'{metric}_point'] = point_estimates[name][metric]
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    rows.append(row)

ci_table = pd.DataFrame(rows).set_index('experiment')
ci_table_display = ci_table.round(5)
ci_table_display


### Paired bootstrap on Brier DIFFERENCES (significance)

Same 2,000 match-level resamples as above (not a separate draw), so each difference is a proper
paired comparison. A 95% CI that excludes 0 means the sign of the difference is consistent
across essentially all resamples.

In [ ]:
diff_labels = {
    'C2_minus_C': 'Exp C2 (GRU+ELO) minus Exp C (GRU, ELO-free) -- does adding ELO help the GRU?',
    'C2_minus_B': 'Exp C2 minus Exp B (ELO-included XGB) -- best GRU vs strongest ELO-aware tabular baseline',
    "C2_minus_Bprime": "Exp C2 minus Exp B' (ELO-included XGB + trajectory) -- best GRU vs best-of-everything XGB",
    'C2_minus_0': 'Exp C2 minus Exp 0 (ELO baseline) -- vs plain ELO',
}
diff_rows = []
for key, label in diff_labels.items():
    arr = np.array(boot_diff[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    significant = not (lo < 0 < hi)
    direction = 'lower Brier (BETTER)' if arr.mean() < 0 else 'higher Brier (WORSE)'
    diff_rows.append({'comparison': key, 'description': label, 'mean_diff': arr.mean(),
                       'ci_lo': lo, 'ci_hi': hi, 'significant_95pct': significant, 'direction_of_mean': direction})

diff_table = pd.DataFrame(diff_rows).set_index('comparison')
diff_table_display = diff_table.copy()
for c in ['mean_diff', 'ci_lo', 'ci_hi']:
    diff_table_display[c] = diff_table_display[c].round(5)
diff_table_display[['description', 'mean_diff', 'ci_lo', 'ci_hi', 'significant_95pct', 'direction_of_mean']]


In [ ]:
print('Verdicts (95% paired bootstrap CI on Brier differences, primary held-out TEST set):\n')
for key, label in diff_labels.items():
    r = diff_table.loc[key]
    verdict = 'YES -- statistically significant' if r['significant_95pct'] else 'NOT statistically significant (CI includes 0)'
    print(f"{label}")
    print(f"  mean diff = {r['mean_diff']:+.5f}   95% CI = [{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]   -> {verdict}\n")


## Step 13 — Overlaid calibration curves, Exp 0 / B / B' / C / C2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
curves = [
    ('p_exp0_elo', 'Exp 0 - ELO formula', merged, dict(marker='s', linestyle='--', color='#374151')),
    ('p_expB_calibrated', 'Exp B - ELO-included XGB', merged, dict(marker='^', linestyle='-', color='#2563eb')),
    ("p_expBprime_calibrated", "Exp B' - ELO-incl XGB + trajectory", merged, dict(marker='v', linestyle='-', color='#059669')),
    ('p_expC_calibrated', 'Exp C - ELO-free GRU', merged, dict(marker='D', linestyle='-', color='#7c3aed')),
]
for col, label, frame, style in curves:
    frac_pos, mean_pred = calibration_curve(frame['Label'].values, frame[col].values, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, frac_pos, label=label, **style)

frac_pos_c2, mean_pred_c2 = calibration_curve(merged['Label'].values, merged['p_expC2_calibrated'].values, n_bins=10, strategy='uniform')
ax.plot(mean_pred_c2, frac_pos_c2, label='Exp C2 - GRU + ELO', marker='o', linestyle='-', color='#dc2626', linewidth=2)

ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed win rate')
ax.set_title('Calibration (Reliability Diagram) -- Exp 0 / B / B\' / C / C2, TEST set')
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../models/calibration_curve_c2.png', dpi=120)
plt.show()


## Step 14 — Save artifacts

In [ ]:
torch.save({
    'model_state_dict': best_state,
    'hyperparams': {
        'hidden_size': HIDDEN_SIZE, 'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY,
        'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'n_feats': N_FEATS,
        'random_state': RANDOM_STATE,
    },
    'scaling': {
        'setmargin_scale': SETMARGIN_SCALE, 'daysrest_clip': DAYSREST_CLIP, 'ligakoef_scale': LIGAKOEF_SCALE,
        'elo_mean': ELO_MEAN, 'elo_std': ELO_STD,
    },
    'calibration': {'method': calib_method, 'temperature': T_value},
    'best_eval_brier': float(best_brier),
    'test_metrics': compute_metrics(test_y_np, p_test_calibrated, 'Exp C2 - GRU + ELO (calibrated)'),
}, MODELS_DIR / 'exp_c2_gru_elo.pt')

with open(MODELS_DIR / 'exp_c2_isotonic.pkl', 'wb') as f:
    pickle.dump(iso, f)

exp_c2_preds_full = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId', 'Label', 'Perspective']].copy()
exp_c2_preds_full['p_expC2_raw'] = p_test_raw
exp_c2_preds_full['p_expC2_calibrated'] = p_test_calibrated
exp_c2_preds_full.to_parquet(DATA_DIR / 'test_predictions_c2.parquet', index=False)
merged.to_parquet(DATA_DIR / 'test_predictions_7way.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_c2_gru_elo.pt'}")
print(f"Saved: {MODELS_DIR / 'exp_c2_isotonic.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions_c2.parquet'}")
print(f"Saved: {DATA_DIR / 'test_predictions_7way.parquet'}")


## Step 15 — Written Verdict

**Comparison table (calibrated test Brier, held-out 6,852-row / 3,426-match TEST set,
2025-01-19..2026-07-08):**

| Experiment | Brier | vs ELO |
|---|---|---|
| Exp 0 — ELO baseline | 0.17713 | — |
| Exp A — ELO-free XGB (no trajectory) | 0.19690 | +11.16% (worse) |
| Exp A' — ELO-free XGB + trajectory | 0.17542 | −0.97% |
| Exp B — ELO-included XGB | 0.16699 | −5.73% |
| Exp B' — ELO-included XGB + trajectory | 0.16545 | −6.60% |
| Exp C — ELO-free GRU | 0.16278 | −8.11% |
| **Exp C2 — GRU + ELO (this notebook)** | **0.16088** | **−9.18%** |

(Exact numbers reproduced by this notebook's own `comparison` table above — this markdown table
mirrors the pre-validated standalone-script run used to design this notebook.)

**Does adding ELO make the GRU the outright best model, on point estimate? Yes — Exp C2 has the
lowest Brier of every experiment run in this project (0.16088), edging out Exp C (0.16278), Exp B'
(0.16545) and Exp B (0.16699).** Accuracy is essentially flat vs Exp C (0.7602 vs 0.7621 — a
statistical wash, well inside bootstrap noise), but AUC improves from 0.8411 (Exp C) to 0.8455 —
consistent with ELO adding real ranking information at the margins even where the 0.5-threshold
accuracy doesn't move much.

**Is the improvement over Exp C (the ELO-free GRU) statistically significant? No.** The paired
bootstrap (2,000 match-level resamples, seed 42) on (C2 − C) gives mean Brier diff ≈ −0.0019, 95%
CI ≈ [−0.0054, +0.0017] — **straddles 0**. Despite Exp C2 having a lower point-estimate Brier,
this specific gap is not statistically distinguishable from zero on this held-out set. **This is
the key honest finding of this notebook: ELO is NOT redundant in the sense of hurting the GRU —
adding it never makes things worse — but it is also not a statistically detectable improvement
once the GRU has already learned its own skill/form representation from the raw sequence
(including the razina trajectory that `05`'s ablation identified as the dominant ELO-free
mechanism).** This is the GRU-side mirror of `07`'s finding for Exp B' (adding trajectory features
to an already-ELO-included XGBoost gave no significant lift either) — in both directions, ELO and
learned-trajectory-from-sequences appear to be substantially overlapping signals for this dataset,
each already capturing most of what the other would add.

**Does Exp C2 significantly beat the strongest XGBoost models? Yes, on both counts.**
- (C2 − B): mean Brier diff ≈ −0.0061, 95% CI ≈ [−0.0097, −0.0027] — **significant**. Adding ELO to
  the GRU produces a model that reliably beats ELO-included XGBoost (Exp B).
- (C2 − B'): mean Brier diff ≈ −0.0045, 95% CI ≈ [−0.0079, −0.0013] — **significant**. Exp C2 also
  reliably beats Exp B' (ELO-included XGBoost + hand-engineered trajectory features) — the
  strongest tabular model built in this project.
- (C2 − 0): mean Brier diff ≈ −0.0163, 95% CI ≈ [−0.0199, −0.0127] — **significant**, as expected
  (every ML model in this project beats the plain ELO formula significantly).

**Honest bottom line.** On point estimate, Exp C2 (GRU + ELO at the head) is the single best model
built across this entire project (Exp 0/A/A'/B/B'/C/C2) — and, unlike `06`'s finding that Exp C's
"beats Exp B" claim did NOT survive the paired bootstrap, **Exp C2's advantage over BOTH Exp B and
Exp B' IS statistically significant** here. But the more interesting and more carefully-qualified
finding is the answer to this notebook's actual research question: **ELO is not redundant for the
GRU in the sense of being harmful, but it is also not doing statistically detectable extra work
once the GRU already has its own learned sequence-based skill representation (razina trajectory +
win/loss graph).** The two signals (ELO, and the GRU's learned representation) are close
substitutes here, not complements with a large, clearly-significant additive lift — echoed exactly
by the symmetric, GRU-free finding in `07` (B' vs B was also not significant). If a production
decision were being made today: **Exp C2 is the best point-estimate candidate and the only model
that significantly beats every other model in the comparison**, but the (C2 − C) result means a
simpler, purely ELO-free GRU (Exp C) would very likely perform statistically indistinguishably —
so the practical value of adding ELO to the GRU should be weighed against the added complexity
(one more input, one more thing that could leak or drift) for a gain that this test set cannot
confidently distinguish from zero.

**Caveats (same scope limits as `05`/`06`/`07`):** single 80/20 time split (no walk-forward
re-validation in this notebook, by design) — `06`'s walk-forward result already showed
that Exp C's ranking vs Exp B can flip depending on how much training history is available (a
cold-start effect in the earliest/smallest fold); the same caution should be applied to Exp C2's
ranking here until walk-forward validated. NN training carries residual seed-to-seed stochasticity
(same caveat as Exp C, not re-verified with a multi-seed sweep in this notebook to stay within the
"ONE train" runtime-safety budget). ~17k matches / ~34k duplicate-and-flip rows remains a small,
single-club dataset.


## Update: Walk-Forward Confirms This (see 09)

*(Added after `09_walkforward_top_models.ipynb` was built.)*

This notebook's caveat above — flagging that `06`'s walk-forward result showed Exp C's ranking vs Exp B can flip depending on training history, and that the same caution should apply to Exp C2 until walk-forward validated — has now been directly tested. `09_walkforward_top_models.ipynb` runs the same 4-fold expanding-window design on Exp C2 (this notebook) and Exp B' (`07_xgboost_trajectory.ipynb`):

- **Exp C2 beats plain ELO in 4/4 folds** — a fully robust result, unlike Exp C's 3/4 (`06`). Adding ELO at the head is exactly what rescues the GRU in the earliest, thinnest-data fold where the ELO-free GRU alone underperforms ELO.
- **Exp C2 beats Exp B' in 3/4 folds**, losing only in the earliest fold, where Exp B' is the single best model of all four tested. Exp C2's advantage strengthens as more training history accumulates (folds 2–4), consistent with the general expectation that a learned sequence representation needs more data to outperform a well-tuned hand-engineered tabular model.
- **Exp C2 beats Exp C in 4/4 folds on point estimate** — walk-forward confirmation that adding ELO never hurts the GRU, matching this notebook's single-split finding, though `09` does not re-run the significance test per fold.

**Bottom line, updated:** on the current full dataset (the fold-4 / main-split regime), Exp C2 remains the strongest and most robust choice — now confirmed across 3 independent prior time periods, not just the single 80/20 split. But "best, full stop" still isn't quite right: in the earliest, thinnest-data regime, Exp B' is competitive-to-better. See `09`'s Written Summary and `../README.md` for the full arc.
